# QuTiP Capability Survey — oriented toward Quantum Reservoir Computing

**Environment:** `qdrug`  ·  **QuTiP:** 5.3.0

QuTiP (Quantum Toolbox in Python) is a library for simulating **open and closed
quantum systems**. Its core object is the `Qobj` — a quantum object wrapping a
(sparse) matrix together with its *dims* (tensor-product structure), so it knows
whether it is a ket, a bra, an operator, or a superoperator.

This notebook is a **map of what QuTiP automates**, grouped by task. Each section
states *what it does for you*, then shows a minimal runnable example. Examples are
biased toward the primitives you need for a **Hamiltonian quantum reservoir**:

| Task | QuTiP does the work in… |
|---|---|
| State preparation | `basis`, `coherent`, `ket2dm`, `tensor`, `rand_dm` |
| Operators / Hamiltonians | `sigmax/y/z`, `qeye`, `tensor`, `destroy`, `jmat` |
| Closed-system evolution | `sesolve`, `mesolve`, `Qobj.expm()`, `propagator` |
| Open-system / dissipation | `mesolve` with collapse operators, `liouvillian` |
| Expectation values (reservoir features) | `expect`, `e_ops` in solvers |
| Reduced states / readout | `ptrace`, `entropy_vn`, `concurrence` |
| Steady state / spectra | `steadystate`, `correlation`, `spectrum` |
| Time-dependent drive | string / function / `QobjEvo` coefficients |

> Run top-to-bottom. Cells are self-contained after the imports cell.

## 0. Imports & conventions

In [1]:
import numpy as np
import qutip as qt
from qutip import (
    basis, coherent, fock, ket2dm, tensor, qeye,
    sigmax, sigmay, sigmaz, sigmap, sigmam, destroy,
    expect, ptrace, mesolve, sesolve, Qobj,
)

print("QuTiP", qt.__version__)

# Single-qubit shorthands used throughout
sx, sy, sz = sigmax(), sigmay(), sigmaz()
I2 = qeye(2)

# Handy: build an operator acting on site `k` of an N-qubit chain
def op_at(op, k, N):
    "Embed single-site operator `op` at position k in an N-site tensor space."
    ops = [I2] * N
    ops[k] = op
    return tensor(ops)


QuTiP 5.3.0


## 1. State preparation

QuTiP gives you constructors for the states you actually use, and the algebra to
combine them. You rarely hand-type a column vector.

- **Computational basis:** `basis(dim, n)` / `fock(dim, n)` → the ket \|n⟩.
- **Superpositions:** just add/normalize `Qobj`s (`(a+b).unit()`).
- **Density matrices:** `ket2dm(psi)`, or `rand_dm(dim)` for a random mixed state.
- **Bosonic modes:** `coherent(N, alpha)`, `thermal_dm`, `fock`.
- **Multipartite:** `tensor(...)` builds the product space and tracks *dims*.

In [2]:
# |0>, |1>, and |+> = (|0>+|1>)/sqrt(2)
zero, one = basis(2, 0), basis(2, 1)
plus = (zero + one).unit()
print("|+> =\n", plus)

# A Bell state |Φ+> = (|00> + |11>)/sqrt(2), built by algebra on the product space
bell = (tensor(zero, zero) + tensor(one, one)).unit()
print("\nBell dims:", bell.dims, " norm:", bell.norm())

# Density matrices: pure via ket2dm, mixed via rand_dm
rho_plus = ket2dm(plus)
rho_mixed = qt.rand_dm(2, seed=0)
print("\npurity(|+>) =", (rho_plus**2).tr().real,
      "  purity(random) =", (rho_mixed**2).tr().real)

# A random product initial state for an N-qubit reservoir
N = 4
psi0 = tensor([qt.rand_ket(2, seed=i) for i in range(N)])
print("\nReservoir init dims:", psi0.dims)


|+> =
 Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.70710678]
 [0.70710678]]

Bell dims: [[2, 2], [1]]  norm: 0.9999999999999999

purity(|+>) = 0.9999999999999996   purity(random) = 0.9167228729986772

Reservoir init dims: [[2, 2, 2, 2], [1]]


## 2. Operators & building a Hamiltonian

Everything is a `Qobj`. Pauli/ladder/number operators are built-ins; you compose
larger Hamiltonians with `tensor` (spatial structure) and `+`/`*`/scalar mult
(algebra). `op_at` (defined above) is the idiom for "σ on site k, identity
elsewhere."

Below: the **transverse-field Ising model (TFIM)** on a chain — a workhorse
reservoir Hamiltonian:

$$H = -J\sum_{k} \sigma^z_k \sigma^z_{k+1} \;-\; h\sum_k \sigma^x_k$$

In [3]:
def tfim_H(N, J=1.0, h=0.5, periodic=False):
    "Transverse-field Ising Hamiltonian on N sites."
    H = 0
    for k in range(N - 1):
        H += -J * op_at(sz, k, N) * op_at(sz, k + 1, N)
    if periodic and N > 2:
        H += -J * op_at(sz, N - 1, N) * op_at(sz, 0, N)
    for k in range(N):
        H += -h * op_at(sx, k, N)
    return H

H = tfim_H(N=4, J=1.0, h=0.5)
print("H is Hermitian:", H.isherm, " shape:", H.shape, " dims:", H.dims)

# Diagonalize: eigenenergies + eigenstates come for free
evals, evecs = H.eigenstates()
print("Ground-state energy:", round(evals[0], 6))
print("Gap E1 - E0:", round(evals[1] - evals[0], 6))


H is Hermitian: True  shape: (16, 16)  dims: [[2, 2, 2, 2], [2, 2, 2, 2]]
Ground-state energy: -3.427034
Gap E1 - E0: 0.094788


## 3. Closed-system (unitary) time evolution

Three levels of automation:

1. **One-shot propagator** — `U = (-1j*H*t).expm()`, then `U*psi0`. Good for a
   single fixed step (e.g. the reservoir's Trotter/analog step you apply per input).
2. **Schrödinger solver** — `sesolve(H, psi0, tlist, e_ops=...)` integrates the ODE
   and can record expectation values along the way.
3. **`propagator(H, t)`** — returns the evolution *operator* (or a list of them),
   handy when you reuse the same map on many inputs.

In [4]:
# (1) Direct matrix exponential — the "apply one reservoir step" primitive
t_step = 1.0
U = (-1j * H * t_step).expm()
psi1 = U * psi0
print("Unitary check ||U^dag U - I||:", (U.dag() * U - qeye(U.dims[0])).norm())

# (2) sesolve with observables recorded at each time
tlist = np.linspace(0, 5, 101)
e_ops = [op_at(sz, k, 4) for k in range(4)]   # <sigma_z> on each site
res = sesolve(H, psi0, tlist, e_ops=e_ops)
print("sesolve recorded", len(res.expect), "observables x", len(tlist), "times")

# (3) propagator: reusable evolution operator for a fixed step
U_prop = qt.propagator(H, t_step)
print("||propagator - expm|| =", round((U_prop - U).norm(), 10),
      "(agree to solver tolerance)")


Unitary check ||U^dag U - I||: 8.54313071832901e-15
sesolve recorded 4 observables x 101 times
||propagator - expm|| = 5.7445e-06 (agree to solver tolerance)


## 4. Expectation values — the reservoir readout features

`expect(op, state)` handles kets *and* density matrices, single operators or
lists, single states or trajectories. In QRC these ⟨O⟩ time series **are** the
reservoir signals you feed to the linear readout.

In [5]:
# Expectation over the whole sesolve trajectory: shape (n_ops, n_times)
sz_traj = np.array(res.expect)      # <sigma_z_k>(t)
print("Feature matrix shape (sites x time):", sz_traj.shape)

# expect also takes a list of ops against one state -> vector of features
feat_ops = [op_at(sz, k, 4) for k in range(4)] + \
           [op_at(sx, k, 4) for k in range(4)]
features = expect(feat_ops, psi1)   # 8-dim feature vector after one step
print("One-step reservoir feature vector:", np.round(features, 4))

# Two-point correlator <sigma_z_0 sigma_z_2> as an engineered feature
zz02 = expect(op_at(sz, 0, 4) * op_at(sz, 2, 4), psi1)
print("<Z0 Z2> =", round(zz02, 6))


Feature matrix shape (sites x time): (4, 101)
One-step reservoir feature vector: [-0.7335 -0.284  -0.4647  0.4721  0.0917 -0.3867  0.1102 -0.4249]
<Z0 Z2> = 0.334014


## 5. Open systems & dissipation — `mesolve`

Real reservoirs leak. `mesolve` integrates the **Lindblad master equation**

$$\dot\rho = -i[H,\rho] + \sum_j \Big(L_j \rho L_j^\dagger - \tfrac12\{L_j^\dagger L_j,\rho\}\Big)$$

You pass **collapse operators** `c_ops` (the $L_j$). Dephasing and amplitude damping
are the two you meet most; both control the reservoir's *fading memory* / echo-state
property.

In [6]:
gamma = 0.15   # damping/dephasing rate
# Amplitude damping on every site (sigma_minus), plus dephasing (sigma_z)
c_ops  = [np.sqrt(gamma)      * op_at(sigmam(), k, 4) for k in range(4)]
c_ops += [np.sqrt(0.5*gamma)  * op_at(sz,       k, 4) for k in range(4)]

rho0 = ket2dm(psi0)
# store_final_state=True so we can ptrace the reduced state in section 6
res_open = mesolve(H, rho0, tlist, c_ops=c_ops,
                   e_ops=[op_at(sz, k, 4) for k in range(4)],
                   options={"store_final_state": True})

print("Open-system <Z0> at t=0, mid, end:",
      np.round([res_open.expect[0][0],
                res_open.expect[0][len(tlist)//2],
                res_open.expect[0][-1]], 4))
print("Damping visibly pulls <Z> toward steady value ->",
      "fading memory is present.")


Open-system <Z0> at t=0, mid, end: [-0.8525 -0.621  -0.602 ]
Damping visibly pulls <Z> toward steady value -> fading memory is present.


## 6. Partial trace & subsystem readout — `ptrace`

`rho.ptrace(sel)` returns the reduced density matrix on the subsystem(s) in `sel`,
tracing out the rest. This is how you read a **local patch** of the reservoir, or
quantify entanglement between reservoir partitions.

In [7]:
rho_final = res_open.final_state           # full 4-qubit density matrix

# Reduced state of site 0 alone (trace out 1,2,3)
rho0_site = rho_final.ptrace(0)
print("Reduced rho of site 0:\n", np.round(rho0_site.full(), 4))
print("Purity of site 0:", round((rho0_site**2).tr().real, 4))

# Reduced state of the (0,1) pair -> a 4x4 density matrix
rho_pair = rho_final.ptrace([0, 1])
print("\nPair (0,1) dims:", rho_pair.dims, " shape:", rho_pair.shape)

# Entanglement diagnostics that ride on ptrace
print("Von Neumann entropy of site 0:",
      round(qt.entropy_vn(rho0_site), 4))
bell_dm = ket2dm((tensor(basis(2,0),basis(2,0))+tensor(basis(2,1),basis(2,1))).unit())
print("Concurrence of a Bell state:", round(qt.concurrence(bell_dm), 4))


Reduced rho of site 0:
 [[0.199 +0.j     0.0108+0.0268j]
 [0.0108-0.0268j 0.801 -0.j    ]]
Purity of site 0: 0.6829

Pair (0,1) dims: [[2, 2], [2, 2]]  shape: (4, 4)
Von Neumann entropy of site 0: 0.4971
Concurrence of a Bell state: 1.0


## 7. Time-dependent Hamiltonians — injecting the input signal

For QRC you usually **drive** the reservoir with your input time series. QuTiP
accepts time-dependent Hamiltonians as `[H0, [H1, coeff]]` where `coeff` is a
Python function `f(t, args)` (or a string, or an interpolated array). Here the
input `u(t)` modulates a transverse field.

In [8]:
H0 = tfim_H(N=4, J=1.0, h=0.0)          # static coupling
Hdrive = sum(op_at(sx, k, 4) for k in range(4))  # field the input couples to

# Input signal u(t): a couple of sinusoids (stand-in for a data stream).
# QuTiP 5's "pythonic" coefficient signature: f(t) with optional named args.
def u_of_t(t):
    return 0.8*np.sin(1.3*t) + 0.4*np.sin(0.5*t)

H_td = [H0, [Hdrive, u_of_t]]           # QuTiP's time-dependent format

res_td = sesolve(H_td, psi0, tlist,
                 e_ops=[op_at(sz, k, 4) for k in range(4)])
driven = np.array(res_td.expect)
print("Driven reservoir feature matrix:", driven.shape)
print("Input is now encoded in the <Z_k>(t) signals ->",
      "these are the QRC design matrix columns.")


Driven reservoir feature matrix: (4, 101)
Input is now encoded in the <Z_k>(t) signals -> these are the QRC design matrix columns.


## 8. Steady states, correlations & spectra

For analysis (not per-input evolution) QuTiP automates:

- `steadystate(H, c_ops)` — the $\dot\rho=0$ solution of the Lindblad equation.
- `correlation_2op_1t` / `correlation_2op_2t` — two-time correlators ⟨A(t)B(0)⟩,
  via the quantum regression theorem.
- `spectrum(...)` — the power spectrum (Fourier transform of the correlator).

These characterize the reservoir's intrinsic timescales / memory.

In [9]:
# Steady state of the dissipative reservoir
rho_ss = qt.steadystate(H, c_ops)
print("Steady-state <Z0>:", round(expect(op_at(sz,0,4), rho_ss), 4))
print("Steady-state purity:", round((rho_ss**2).tr().real, 4))

# Two-time autocorrelation of a single-site observable -> memory timescale
taus = np.linspace(0, 10, 200)
corr = qt.correlation_2op_2t(H, rho0, [0], taus,
                             c_ops, op_at(sz,0,4), op_at(sz,0,4))[0]
print("Autocorr <Z0(0) Z0(tau)>: first 3 =", np.round(corr[:3].real, 4))
print("Decay of this correlator ~ reservoir memory length.")


Steady-state <Z0>: -0.5795
Steady-state purity: 0.3032
Autocorr <Z0(0) Z0(tau)>: first 3 = [1.     0.9976 0.9928]
Decay of this correlator ~ reservoir memory length.


## 9. Quick reference — "I want to… → call this"

| I want to… | QuTiP call |
|---|---|
| Make \|n⟩ / \|α⟩ | `basis`, `fock` / `coherent` |
| Pure→density / random mixed | `ket2dm` / `rand_dm` |
| Combine subsystems | `tensor` |
| Pauli / ladder / number ops | `sigmax/y/z`, `sigmap/m`, `destroy`, `num` |
| Diagonalize H | `H.eigenstates()`, `H.eigenenergies()` |
| One unitary step | `(-1j*H*t).expm()` or `propagator(H, t)` |
| Schrödinger evolution + observables | `sesolve(H, ψ, t, e_ops=…)` |
| Lindblad / dissipation | `mesolve(H, ρ, t, c_ops=…, e_ops=…)` |
| Build the Liouvillian directly | `liouvillian(H, c_ops)` |
| Expectation value(s) | `expect(op(s), state(s))` |
| Reduced density matrix | `state.ptrace(sel)` |
| Entropy / entanglement | `entropy_vn`, `concurrence`, `negativity` |
| Steady state | `steadystate(H, c_ops)` |
| Two-time correlator / spectrum | `correlation_2op_2t`, `spectrum` |
| Time-dependent H (input drive) | `[H0, [H1, f(t,args)]]`, `QobjEvo` |
| Fidelity / distance | `fidelity`, `tracedist` |

**QRC takeaway:** a Hamiltonian reservoir is just — *pick H* (§2), *encode input*
into state or drive (§1, §7), *evolve* (§3/§5), *read* ⟨O⟩ or `ptrace` features
(§4/§6), then a classical linear readout on those features. QuTiP automates every
quantum step in that loop.

In [10]:
import re
PAULI_OPERATORS = {
    "I": qeye(2),
    "X": sigmax(),
    "Y": sigmay(),
    "Z": sigmaz(),
}


def pauli_word_to_matrix(word: str) -> Qobj:
    """
    Convert a Pauli word such as 'XXII' into the QuTiP operator
        X ⊗ X ⊗ I ⊗ I
    Lowercase letters and whitespace are accepted.
    """
    word = word.replace(" ", "").upper()

    if not word:
        raise ValueError("The Pauli word cannot be empty.")

    invalid = set(word) - set(PAULI_OPERATORS)
    if invalid:
        raise ValueError(
            f"Invalid Pauli characters: {sorted(invalid)}. "
            "Only I, X, Y, and Z are allowed."
        )

    return tensor([PAULI_OPERATORS[character] for character in word])


def pauli_expression_to_matrix(expression: str) -> Qobj:
    """
    Convert a sum of Pauli words into a QuTiP operator.

    Examples
    --------
    'XXII + zzII'
    'XX - YY + ZZ'
    '2*XXII - 0.5*ZZII'
    """
    expression = expression.replace(" ", "")

    if not expression:
        raise ValueError("The expression cannot be empty.")

    # Ensure the first term has an explicit sign.
    if expression[0] not in "+-":
        expression = "+" + expression

    raw_terms = re.findall(r"([+-])([^+-]+)", expression)

    parsed_terms = []
    number_of_qubits = None

    for sign, term in raw_terms:
        match = re.fullmatch(
            r"(?:(\d+(?:\.\d*)?|\.\d+)\*)?([IXYZixyz]+)",
            term,
        )

        if match is None:
            raise ValueError(f"Could not parse term: {sign}{term}")

        coefficient_text, pauli_word = match.groups()
        coefficient = float(coefficient_text) if coefficient_text else 1.0

        if sign == "-":
            coefficient *= -1

        pauli_word = pauli_word.upper()

        if number_of_qubits is None:
            number_of_qubits = len(pauli_word)
        elif len(pauli_word) != number_of_qubits:
            raise ValueError(
                "All Pauli words must have the same length. "
                f"Expected {number_of_qubits}, got {len(pauli_word)} "
                f"for '{pauli_word}'."
            )

        parsed_terms.append(
            coefficient * pauli_word_to_matrix(pauli_word)
        )

    if not parsed_terms:
        raise ValueError("No valid Pauli terms were found.")

    result = parsed_terms[0]

    for term in parsed_terms[1:]:
        result = result + term

    return result


In [11]:
from qutip import Qobj


def TFIM_to_matrix(num_spin: int, params: dict) -> Qobj:
    """
    params:
        Dictionary containing:
            params["J"] : nearest-neighbour interaction strength
            params["h"] : transverse-field strength
    """
    if num_spin < 1:
        raise ValueError("num_spin must be at least 1.")

    J = params.get("J", 1.0)
    h = params.get("h", 1.0)
    periodic = params.get("periodic", False)

    # Start from the zero operator with the correct dimensions.
    H = 0 * pauli_word_to_matrix("I" * num_spin)

    # Nearest-neighbour ZZ terms.
    for i in range(num_spin - 1):
        pauli_zzterm = ["I"] * num_spin
        pauli_zzterm[i] = "Z"
        pauli_zzterm[i + 1] = "Z"

        pauli_word = "".join(pauli_zzterm)
        H += -J * pauli_word_to_matrix(pauli_word)

    # Optional interaction between the last and first spins.
    if periodic and num_spin > 2:
        pauli_zzterm = ["I"] * num_spin
        pauli_zzterm[-1] = "Z"
        pauli_zzterm[0] = "Z"

        H += -J * pauli_word_to_matrix("".join(pauli_zzterm))

    # Transverse-field X terms.
    for i in range(num_spin):
        pauli_xterm = ["I"] * num_spin
        pauli_xterm[i] = "X"

        pauli_word = "".join(pauli_xterm)
        H += -h * pauli_word_to_matrix(pauli_word)

    return H

In [12]:
params = {
    "J": 1.0,
    "h": 0.5,
    "periodic": False,
}

H = TFIM_to_matrix(num_spin=7, params=params)

display(H)
print(H.shape)   # (16, 16)

Quantum object: dims=[[2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2]], shape=(128, 128), type='oper', dtype=CSR, isherm=True
Qobj data =
[[-6.  -0.5 -0.5 ...  0.   0.   0. ]
 [-0.5 -4.   0.  ...  0.   0.   0. ]
 [-0.5  0.  -2.  ...  0.   0.   0. ]
 ...
 [ 0.   0.   0.  ... -2.   0.  -0.5]
 [ 0.   0.   0.  ...  0.  -4.  -0.5]
 [ 0.   0.   0.  ... -0.5 -0.5 -6. ]]

(128, 128)
